In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gc
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
    median_absolute_error,
    explained_variance_score,
)
from xgboost import XGBRegressor

In [27]:
df = pd.read_parquet("data/train.parquet", engine="pyarrow")
df

,payload,total_distance_m,wind_speed_mean,wind_speed_std,wind_x_mean,wind_y_mean,velocity_mag_mean,velocity_mag_max,acceleration_mag_mean,acceleration_mag_std,battery_needed
0,0.0,76.894875,3.898058,1.952675,0.973186,-1.246141,2.676774,6.261616,9.842870,0.466372,1000.743935
1,0.0,125.616211,3.522941,2.159456,0.088843,-1.886357,2.387432,7.676739,9.881874,0.628406,1205.255874
2,0.0,71.228093,4.581182,3.335733,-0.696895,-1.778235,3.110644,7.213987,9.902090,0.545290,789.073853
3,0.0,78.000098,4.596319,3.438072,-0.351732,-1.389653,3.165734,9.425537,9.900368,0.559073,687.813976
4,0.0,71.103672,3.333910,2.247522,0.062744,-1.257258,2.126439,4.900079,9.817243,0.341981,920.070980
...,...,...,...,...,...,...,...,...,...,...,...
204,500.0,59.757821,4.139878,3.885389,0.336196,-0.676045,3.336389,8.788151,9.850903,0.420902,760.480643
205,500.0,61.401245,4.392581,4.332293,0.252633,-0.635730,3.402998,10.553163,9.862172,0.471081,738.620356
206,500.0,64.842379,5.524651,4.029744,-0.785881,-1.457703,3.680124,10.579715,9.847796,0.529103,741.451254
207,500.0,197.253052,4.686967,3.826570,-0.546397,-1.806563,2.907930,10.376503,9.829065,0.456918,887.198699


In [25]:
del df
gc.collect()
%whos

Variable                         Type              Data/Info
------------------------------------------------------------
ElasticNet                       ABCMeta           <class 'sklearn.linear_mo<...>nate_descent.ElasticNet'>
LinearRegression                 ABCMeta           <class 'sklearn.linear_mo<...>._base.LinearRegression'>
SVR                              ABCMeta           <class 'sklearn.svm._classes.SVR'>
StandardScaler                   type              <class 'sklearn.preproces<...>ng._data.StandardScaler'>
X                                DataFrame         Shape: (209, 10)
XGBRegressor                     type              <class 'xgboost.sklearn.XGBRegressor'>
explained_variance_score         function          <function explained_varia<...>_score at 0x7fdbe36c9440>
gc                               module            <module 'gc' (built-in)>
k                                str               ExplainedVariance
mean_absolute_error              function          <function 

In [28]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 209 entries, 0 to 208
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   payload                209 non-null    float64
 1   total_distance_m       209 non-null    float64
 2   wind_speed_mean        209 non-null    float64
 3   wind_speed_std         209 non-null    float64
 4   wind_x_mean            209 non-null    float64
 5   wind_y_mean            209 non-null    float64
 6   velocity_mag_mean      209 non-null    float64
 7   velocity_mag_max       209 non-null    float64
 8   acceleration_mag_mean  209 non-null    float64
 9   acceleration_mag_std   209 non-null    float64
 10  battery_needed         209 non-null    float64
dtypes: float64(11)
memory usage: 18.1 KB


In [29]:
X = df
y = X.pop("battery_needed")

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size= 0.2, random_state= 42)

sc = StandardScaler()

x_train_sc = sc.fit_transform(x_train)
x_test_sc = sc.transform(x_test)

In [30]:
x_train_sc.shape, y_train.shape

((167, 10), (167,))

In [31]:
model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
model.fit(x_train_sc, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [32]:
xgb_pred = model.predict(x_test_sc)

print('xgb')
# print(f'r2  : {r2_score(y_test, xgb_pred)}')
# print(f'RMSE: {root_mean_squared_error(y_test, xgb_pred)}')
metrics = {
    "MAE": mean_absolute_error(y_test, xgb_pred),
    "MSE": mean_squared_error(y_test, xgb_pred),
    "RMSE": root_mean_squared_error(y_test, xgb_pred),
    "R2": r2_score(y_test, xgb_pred),
    "MAPE": mean_absolute_percentage_error(y_test, xgb_pred),
    "MedianAE": median_absolute_error(y_test, xgb_pred),
    "ExplainedVariance": explained_variance_score(y_test, xgb_pred),
}

for k, v in metrics.items():
    print(f"{k:<20}: {v:.6f}")

xgb
MAE                 : 60.193131
MSE                 : 7782.869656
RMSE                : 88.220574
R2                  : 0.863750
MAPE                : 0.076948
MedianAE            : 35.480735
ExplainedVariance   : 0.864070


In [33]:
pd.set_option("display.float_format", "{:.10f}".format)
comparison = pd.DataFrame({
    "Actual": y_test,
    "Predicted": xgb_pred,
})

comparison["delta"] = comparison["Actual"] - comparison["Predicted"]

comparison

,Actual,Predicted,delta
30,928.6805844937,893.2150268555,35.4655576383
171,715.2303003176,699.6381225586,15.5921777590
84,577.4204049518,759.1337890625,-181.7133841107
198,763.3566236314,785.7545776367,-22.3979540053
60,639.9512120362,654.2692871094,-14.3180750732
155,50.4291561514,29.1322879791,21.2968681723
45,1293.8475577725,1317.8518066406,-24.0042488682
181,933.3797536328,869.3434448242,64.0363088085
9,588.9331596962,656.0902709961,-67.1571112998
195,874.6278060942,910.1237182617,-35.4959121675
